In [ ]:
import os
import time
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import torch
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F
import uvicorn
import threading

In [ ]:
class EmbedRequest(BaseModel):
    texts: List[str]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device=device)

In [ ]:
app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/embed")
def embed(request: EmbedRequest):
    prefixed_texts = ["search_document: " + t for t in request.texts]
    embeddings = model.encode(
        prefixed_texts,
        batch_size=32,
        show_progress_bar=False
    )

    return {"embeddings": embeddings.tolist()}

In [ ]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=8002)

threading.Thread(target=run).start()

INFO:     Started server process [438]


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 121856 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.2.0) over (2026.2.0) ...
Setting up cloudflared (2026.2.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!cloudflared tunnel --url http://localhost:8002

2026-03-05T09:33:32Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-05T09:33:32Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-05T09:33:45Z INF +--------------------------------------------------------------------------------------------+
2026-03-05T09:33:45Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-05T09:33:45Z INF |  https://talked-trivia-remedies-zealand.trycloudflare.

In [ ]:
!pkill -f uvicorn